In [3]:
import os
import shutil
from pathlib import Path

In [11]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

for dataset in Path("/kaggle/input").iterdir():
    imgs = [f for f in dataset.rglob("*") if f.suffix.lower() in IMG_EXTS]
    print(f"{dataset.name}: {len(imgs)} images")

datasets: 3986 images


In [12]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
input_dir = Path("/kaggle/input")

total_imgs = 0
total_labels = 0
no_label = []

for img in input_dir.rglob("*"):
    if img.suffix.lower() not in IMG_EXTS:
        continue
    total_imgs += 1
    label = img.with_suffix(".txt")
    if not label.exists():
        label = img.parent.parent / "labels" / (img.stem + ".txt")
    if label.exists():
        total_labels += 1
    else:
        no_label.append(img.name)

print(f"Total images     : {total_imgs}")
print(f"Images with label: {total_labels}")
print(f"Images no label  : {len(no_label)}")

Total images     : 3986
Images with label: 3986
Images no label  : 0


In [5]:
INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/p1_dataset")
PREFIX     = "pothole_"
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp"}

In [6]:
print("Step 1: Datasets are auto-mounted by Kaggle — no extraction needed.")
print(f"  Scanning: {INPUT_DIR}")
print()

Step 1: Datasets are auto-mounted by Kaggle — no extraction needed.
  Scanning: /kaggle/input



In [7]:
print("Step 2: YOLOv8 format confirmed — no conversion needed.")
print()

Step 2: YOLOv8 format confirmed — no conversion needed.



In [8]:
print("Step 3: Class IDs — 0: pothole  |  1: road_crack — no remapping needed.")
print()

Step 3: Class IDs — 0: pothole  |  1: road_crack — no remapping needed.



In [10]:
print("Step 4: Renaming files with 'pothole_' prefix and copying to output...")
 
images_out = OUTPUT_DIR / "images"
labels_out = OUTPUT_DIR / "labels"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)
 
copied  = 0
skipped = 0
 
for img_path in sorted(INPUT_DIR.rglob("*")):
    if img_path.suffix.lower() not in IMG_EXTS:
        continue
 
    # Try to find matching label — same folder or sibling labels/ folder
    label_path = img_path.with_suffix(".txt")
    if not label_path.exists():
        label_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
 
    new_stem     = f"{PREFIX}{img_path.stem}"
    new_img_name = new_stem + img_path.suffix.lower()
    new_lbl_name = new_stem + ".txt"
 
    shutil.copy2(img_path, images_out / new_img_name)
 
    if label_path.exists():
        shutil.copy2(label_path, labels_out / new_lbl_name)
        copied += 1
    else:
        skipped += 1
        print(f"  [WARN] No label for: {img_path.name}")
 
print(f"  Copied: {copied} image+label pairs  |  {skipped} images without labels")
print()
 

Step 4: Renaming files with 'pothole_' prefix and copying to output...
  Copied: 3986 image+label pairs  |  0 images without labels



In [13]:
print("Step 5: Verifying image <-> label matches...")
 
all_images = {p.stem for p in images_out.glob("*") if p.suffix.lower() in IMG_EXTS}
all_labels = {p.stem for p in labels_out.glob("*.txt")}
 
missing_labels = all_images - all_labels
missing_images = all_labels - all_images
 
if not missing_labels:
    print("  [OK] Every image has a matching label.")
else:
    print(f"  [WARN] {len(missing_labels)} image(s) missing labels:")
    for name in sorted(missing_labels):
        print(f"    - {name}")
 
if not missing_images:
    print("  [OK] Every label has a matching image.")
else:
    print(f"  [WARN] {len(missing_images)} label(s) missing images:")
    for name in sorted(missing_images):
        print(f"    - {name}")
 
print(f"\n  Total images : {len(all_images)}")
print(f"  Total labels : {len(all_labels)}")
print()
 

Step 5: Verifying image <-> label matches...
  [OK] Every image has a matching label.
  [OK] Every label has a matching image.

  Total images : 3986
  Total labels : 3986



In [14]:
print("=" * 50)
print("Step 6: Done! Output is ready.")
print(f"  Folder : {OUTPUT_DIR}")
print(f"  images/: {len(list(images_out.glob('*')))} files")
print(f"  labels/: {len(list(labels_out.glob('*.txt')))} files")
print("=" * 50)

Step 6: Done! Output is ready.
  Folder : /kaggle/working/p1_dataset
  images/: 3986 files
  labels/: 3986 files


In [15]:
from pathlib import Path

labels_dir = Path("/kaggle/working/p1_dataset/labels")

class_map = {0: "pothole", 1: "road_crack"}
class_counts = {0: 0, 1: 0}
unknown_classes = set()

for label_file in labels_dir.glob("*.txt"):
    with open(label_file, "r") as f:
        for line in f:
            class_id = int(line.strip().split()[0])
            if class_id in class_counts:
                class_counts[class_id] += 1
            else:
                unknown_classes.add(class_id)

print("Class Mapping Verification:")
print("=" * 30)
for class_id, class_name in class_map.items():
    print(f"  Class {class_id} ({class_name}): {class_counts[class_id]} instances")

if unknown_classes:
    print(f"\n  [WARN] Unknown class IDs found: {unknown_classes}")
else:
    print("\n  [OK] No unknown classes found!")

Class Mapping Verification:
  Class 0 (pothole): 8598 instances
  Class 1 (road_crack): 2519 instances

  [WARN] Unknown class IDs found: {2}


In [16]:
import shutil
from pathlib import Path

# Zip the output folder
shutil.make_archive(
    "/kaggle/working/p1_dataset",  # output zip name
    "zip",                          # format
    "/kaggle/working",              # root dir
    "p1_dataset"                    # folder to zip
)

print("Zipped! Ready to download → p1_dataset.zip")

Zipped! Ready to download → p1_dataset.zip
